# Pipeline de Análisis Exploratorio — Heart Disease Dataset
**Proyecto Final — Curso 1: Git & GitHub para Data Science**  
Docente: Ing. Marco Mayta
Instituto de analítica y negocios — Lima, Perú

---
## Instalación de dependencias
Ejecuta esta celda una sola vez antes de comenzar.

In [ ]:
# Ejecutar una sola vez
%pip install ucimlrepo pandas matplotlib seaborn --quiet

---
## FASE 1 — Carga, inspección, limpieza y análisis exploratorio

Cada sección de esta fase corresponde a un commit en tu repositorio.  
**No hagas un solo commit con todo. Commitea sección por sección.**

### Sección 1 — Carga del dataset
*Después de completar esta celda → primer commit de código*

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import os

# Descargar el dataset Heart Disease de UCI (Cleveland, id=45)
dataset = fetch_ucirepo(id=45)

# Extraer features y variable objetivo
X = dataset.data.features
y = dataset.data.targets

# Unir en un solo DataFrame
df = pd.concat([X, y], axis=1)

# Renombrar la columna objetivo
df.rename(columns={"num": "enfermedad"}, inplace=True)

# Convertir a variable binaria: 0 = sano, 1 = enfermo
df["enfermedad"] = (df["enfermedad"] > 0).astype(int)

# Guardar para no descargar en cada ejecución
os.makedirs("data", exist_ok=True)
df.to_csv("data/heart.csv", index=False)

print("Dataset descargado correctamente.")
print(f"Forma: {df.shape}")
print(f"Columnas: {list(df.columns)}")

### Sección 2 — Inspección inicial
*Después de completar esta celda → segundo commit de código*

In [ ]:
df = pd.read_csv("data/heart.csv")

print("--- Primeras 5 filas ---")
print(df.head())

In [ ]:
print("--- Tipos de datos ---")
print(df.dtypes)

In [ ]:
print("--- Estadísticas descriptivas ---")
df.describe()

In [ ]:
print("--- Valores nulos por columna ---")
print(df.isnull().sum())

print("\n--- Distribución de la variable objetivo ---")
conteo = df["enfermedad"].value_counts()
print(conteo)
print(f"\nProporción de enfermos: {conteo[1] / len(df):.2%}")

In [ ]:
print("--- Columnas con pocos valores únicos (posibles categóricas) ---")
for col in df.columns:
    if df[col].nunique() <= 6:
        print(f"  {col}: {sorted(df[col].unique())}")

### Sección 3 — Limpieza de datos
*Después de completar esta celda → tercer commit de código*

In [ ]:
df = pd.read_csv("data/heart.csv")

print(f"Filas antes de limpieza: {len(df)}")

# Verificar y eliminar duplicados
n_dup = df.duplicated().sum()
print(f"Duplicados encontrados: {n_dup}")
if n_dup > 0:
    df.drop_duplicates(inplace=True)

In [ ]:
# Verificar rangos inválidos
print("Registros con edad fuera de rango (20-80):",
      len(df[(df["age"] < 20) | (df["age"] > 80)]))

print("Registros con presión arterial = 0:",
      len(df[df["trestbps"] == 0]))

print("Registros con colesterol = 0:",
      len(df[df["chol"] == 0]))

In [ ]:
# Reemplazar ceros inválidos con la mediana de la columna
for col in ["trestbps", "chol"]:
    n_ceros = (df[col] == 0).sum()
    if n_ceros > 0:
        mediana = df[df[col] != 0][col].median()
        df[col] = df[col].replace(0, mediana)
        print(f"  '{col}': {n_ceros} ceros → mediana={mediana}")
    else:
        print(f"  '{col}': sin ceros inválidos.")

# Guardar dataset limpio
df.to_csv("data/heart_limpio.csv", index=False)
print(f"\nDataset limpio guardado. Filas finales: {len(df)}")

### Sección 4 — Análisis exploratorio (EDA)
*Después de completar esta celda → cuarto commit de código*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("outputs", exist_ok=True)
df = pd.read_csv("data/heart_limpio.csv")

In [ ]:
# Figura 1 — Distribución de edad por condición
fig, ax = plt.subplots(figsize=(8, 5))
for cond, label, color in [(0, "Sano", "#28a745"), (1, "Enfermo", "#d16a30")]:
    df[df["enfermedad"] == cond]["age"].plot(
        kind="hist", bins=20, alpha=0.6, ax=ax,
        label=label, color=color
    )
ax.set_title("Distribución de edad según condición cardíaca")
ax.set_xlabel("Edad")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/dist_edad.png", dpi=100)
plt.show()
print("Guardada: outputs/dist_edad.png")

In [ ]:
# Figura 2 — Proporción de enfermos por sexo
tabla = df.groupby("sex")["enfermedad"].mean().reset_index()
tabla["sex"] = tabla["sex"].map({0: "Mujer", 1: "Hombre"})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tabla["sex"], tabla["enfermedad"],
       color=["#28a745", "#24292e"], width=0.4)
ax.set_title("Proporción de enfermedad cardíaca por sexo")
ax.set_ylabel("Proporción")
ax.set_ylim(0, 1)
for i, v in enumerate(tabla["enfermedad"]):
    ax.text(i, v + 0.02, f"{v:.1%}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("outputs/proporcion_sexo.png", dpi=100)
plt.show()
print("Guardada: outputs/proporcion_sexo.png")

In [ ]:
# Figura 3 — Matriz de correlación
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de correlación — variables cardiovasculares")
plt.tight_layout()
plt.savefig("outputs/correlacion.png", dpi=100)
plt.show()
print("Guardada: outputs/correlacion.png")

In [ ]:
# Tabla resumen por condición
resumen = df.groupby("enfermedad")[["age","trestbps","chol","thalach"]].mean()
resumen.index = resumen.index.map({0: "Sano", 1: "Enfermo"})
print("--- Estadísticas medias por condición ---")
resumen.round(2)

---
## FASE 2 — Corrección del historial

Esta celda representa el cambio que vas a commitear con un mensaje **malo** a propósito  
y luego corregir usando `git reset`. Lee las instrucciones de la guía antes de ejecutar.

In [ ]:
# Agrega al final del README.md el resultado clave del EDA.
# Este cambio es el que vas a commitear mal y luego corregir.

prop_enfermos = df["enfermedad"].mean()

linea = (
    f"\n## Resultado clave\n"
    f"El {prop_enfermos:.1%} de los pacientes del dataset "
    f"presenta enfermedad cardíaca.\n"
)

with open("README.md", "a", encoding="utf-8") as f:
    f.write(linea)

print("README.md actualizado con el resultado del EDA.")
print(linea)

---
## FASE 3 — Rama `feature/visualizaciones`

Crea la rama con `git checkout -b feature/visualizaciones` **antes** de ejecutar esta celda.  
Esta sección agrega una figura nueva al análisis.

In [ ]:
df = pd.read_csv("data/heart_limpio.csv")

# Figura 4 — Boxplot de colesterol por condición
fig, ax = plt.subplots(figsize=(7, 5))
df.boxplot(column="chol", by="enfermedad", ax=ax,
           patch_artist=True,
           boxprops=dict(facecolor="#d16a30", color="#24292e"),
           medianprops=dict(color="white", linewidth=2))
ax.set_title("Colesterol según condición cardíaca")
ax.set_xlabel("Condición (0 = Sano, 1 = Enfermo)")
ax.set_ylabel("Colesterol (mg/dl)")
plt.suptitle("")
plt.tight_layout()
plt.savefig("outputs/boxplot_colesterol.png", dpi=100)
plt.show()
print("Guardada: outputs/boxplot_colesterol.png")

---
## FASE 3 — Rama `feature/estadisticas`

Vuelve a `main` con `git checkout main`, luego crea la rama con  
`git checkout -b feature/estadisticas` **antes** de ejecutar esta celda.

In [ ]:
df = pd.read_csv("data/heart_limpio.csv")

# Crear grupos de edad
bins   = [20, 40, 55, 70, 90]
labels = ["20-40", "41-55", "56-70", "71+"]
df["grupo_edad"] = pd.cut(df["age"], bins=bins,
                           labels=labels, right=True)

print("=" * 55)
print("RESUMEN ESTADÍSTICO POR GRUPO DE EDAD")
print("=" * 55)

In [ ]:
# Proporción de enfermos por grupo de edad
print("--- Proporción de enfermedad por grupo ---")
prop = df.groupby("grupo_edad", observed=True)["enfermedad"].mean()
for grupo, valor in prop.items():
    barra = "█" * int(valor * 20)
    print(f"  {grupo:6s}: {valor:.2%}  {barra}")

In [ ]:
# Estadísticas de presión y colesterol por grupo
print("--- Presión arterial media por grupo ---")
print(df.groupby("grupo_edad", observed=True)["trestbps"].mean().round(1))

print("\n--- Colesterol medio por grupo ---")
print(df.groupby("grupo_edad", observed=True)["chol"].mean().round(1))

In [ ]:
# Guardar resumen completo
resumen = df.groupby("grupo_edad", observed=True).agg(
    n_registros      = ("age",        "count"),
    edad_media       = ("age",        "mean"),
    prop_enfermos    = ("enfermedad", "mean"),
    presion_media    = ("trestbps",   "mean"),
    colesterol_medio = ("chol",       "mean")
).round(2)

resumen.to_csv("outputs/resumen_por_grupo.csv")
print("Guardado: outputs/resumen_por_grupo.csv")
resumen